<a href="https://colab.research.google.com/github/cout-angela/projectP_imaging/blob/main/group_p_template_esempio_ippy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group P – Computational Imaging Project
**Task:** Motion Deblur + Denoising  
**Dataset:** LSUN Church  
**Image size:** 256×256

---

## Directory Structure

Before running this notebook, make sure your project folder is organized as follows:

```
group_p/
├── IPPy/
│   ├── __init__.py
│   ├── operators.py
│   ├── solvers.py
│   ├── nn/
│   └── utilities/
├── data/
│   ├── train/
│   ├── val/
│   └── test/
├── model_weights/
├── results/
└── group_p_template_ippy.ipynb  ← this notebook
```

Clone IPPy with:
```bash
git clone https://github.com/devangelista2/IPPy.git
```
Then move the inner `IPPy/` folder to your project root (see [environment setup](https://devangelista2.github.io/computational-imaging/2025-26/intro/environment-setup.html)).

---
## 0 – Imports & Environment Check

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# ── IPPy ─────────────────────────────────────────────────────────────────────
# IPPy is not installed as a package: it lives as a subfolder in the project
# root. We add the root to sys.path so Python can import it directly.
ROOT = Path.cwd()          # adjust if the notebook is in a subdirectory
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import IPPy.operators as ops
import IPPy.solvers   as solvers
import IPPy.utilities as utils   # adjust sub-module names to the actual IPPy version

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"IPPy found at: {ROOT / 'IPPy'}")

---
## 1 – Configuration
All experiment hyper-parameters live here so they are easy to change.

In [ ]:
# ── Image & dataset ──────────────────────────────────────────────────────────
IMG_SIZE   = 256
DATA_DIR   = ROOT / "data"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# ── Degradation parameters (from project spec) ────────────────────────────────
KERNEL_SIZE   = 9
MOTION_ANGLE  = 45        # TODO: choose your preferred angle
NOISE_LEVELS  = [0.005, 0.01, 0.05, 0.1]

# ── Metrics storage ──────────────────────────────────────────────────────────
results = {}              # filled in each method section

---
## 2 – Dataset Loading & Preprocessing
> **TODO:** implement the dataset loading pipeline for LSUN Church.  
> - Resize to 256×256  
> - Normalize to [0, 1] (float32)  
> - Split into train / val / test  
> - Document every preprocessing choice

In [ ]:
# TODO: load LSUN Church dataset
# Reference: https://github.com/fyu/lsun
#
# Expected output tensors (float32, values in [0,1]):
#   x_train : (N_train, 1, 256, 256)  – grayscale or (N_train, 3, 256, 256) for RGB
#   x_val   : (N_val,   1, 256, 256)
#   x_test  : (N_test,  1, 256, 256)

x_train = None   # placeholder
x_val   = None   # placeholder
x_test  = None   # placeholder

---
## 3 – Forward Operator (Degradation) with IPPy

We use **IPPy** to build the motion-blur + noise forward operator.  
All methods will receive *exactly the same* degraded observations.

In [ ]:
# ── Build the motion-blur operator with IPPy ─────────────────────────────────
# IPPy.operators typically exposes linear operators as objects with
# .fwd(x) / .adj(y) methods (or __call__ / .T depending on the version).
# Adjust the constructor call to match the actual IPPy API.
#
# Example (adapt as needed):
#   K = ops.MotionBlur(size=KERNEL_SIZE, angle=MOTION_ANGLE, img_size=IMG_SIZE)

# TODO: instantiate the IPPy motion-blur operator
K = None   # placeholder – replace with the correct IPPy call


def degrade(x_clean: np.ndarray, noise_level: float) -> np.ndarray:
    """
    Apply the forward degradation model:  y = K(x) + noise.

    Parameters
    ----------
    x_clean : np.ndarray
        Clean image, shape (H, W) or (C, H, W), float32 in [0, 1].
    noise_level : float
        Standard deviation of the additive Gaussian noise.

    Returns
    -------
    y : np.ndarray
        Degraded observation, same shape as x_clean.
    """
    # TODO: call K.fwd(x_clean) (or equivalent IPPy syntax) then add noise
    y_blurred = None   # K.fwd(x_clean)
    noise     = noise_level * np.random.randn(*x_clean.shape).astype(np.float32)
    y         = y_blurred + noise
    return np.clip(y, 0.0, 1.0)


# ── Generate degraded test observations (shared across all methods) ──────────
# y_test[sigma] = degraded version of x_test at noise level sigma
y_test = {}
for sigma in NOISE_LEVELS:
    # TODO: call degrade on x_test once IPPy operator and dataset are ready
    y_test[sigma] = None   # placeholder

### 3.1 – Visualise a Degraded Sample

In [ ]:
# TODO: visualise one clean image and its degraded counterparts at each noise level
# Example skeleton (uncomment and adapt once data is loaded):
#
# fig, axes = plt.subplots(1, len(NOISE_LEVELS) + 1, figsize=(16, 4))
# axes[0].imshow(x_test[0].squeeze(), cmap="gray"); axes[0].set_title("Clean")
# for i, sigma in enumerate(NOISE_LEVELS, start=1):
#     axes[i].imshow(y_test[sigma][0].squeeze(), cmap="gray")
#     axes[i].set_title(f"σ = {sigma}")
# plt.tight_layout(); plt.show()

---
## 4 – Helper: Metrics

A simple utility to compute PSNR and SSIM, consistent across all methods.

In [ ]:
def compute_metrics(x_gt: np.ndarray, x_rec: np.ndarray) -> dict:
    """
    Compute PSNR and SSIM between ground truth and reconstruction.

    Both arrays should be float32 in [0, 1], shape (H, W) or (C, H, W).
    Multi-channel inputs are handled via channel_axis.
    """
    x_gt  = x_gt.squeeze()
    x_rec = x_rec.squeeze()
    channel_axis = 0 if x_gt.ndim == 3 else None

    psnr_val = psnr(x_gt, x_rec, data_range=1.0)
    ssim_val = ssim(x_gt, x_rec, data_range=1.0, channel_axis=channel_axis)
    return {"PSNR": psnr_val, "SSIM": ssim_val}


def evaluate_on_test(
    method_name: str,
    reconstruct_fn,          # callable: (y_degraded, sigma) -> x_rec
    noise_levels: list,
    x_test,
    y_test: dict,
) -> dict:
    """
    Run reconstruct_fn on every test image and noise level; return mean metrics.
    """
    metrics = {sigma: {"PSNR": [], "SSIM": []} for sigma in noise_levels}
    for sigma in noise_levels:
        for i in range(len(x_test)):
            x_rec = reconstruct_fn(y_test[sigma][i], sigma)
            m = compute_metrics(x_test[i], x_rec)
            metrics[sigma]["PSNR"].append(m["PSNR"])
            metrics[sigma]["SSIM"].append(m["SSIM"])
        metrics[sigma]["PSNR"] = float(np.mean(metrics[sigma]["PSNR"]))
        metrics[sigma]["SSIM"] = float(np.mean(metrics[sigma]["SSIM"]))
        print(f"[{method_name}] σ={sigma:.3f}  "
              f"PSNR={metrics[sigma]['PSNR']:.2f} dB  "
              f"SSIM={metrics[sigma]['SSIM']:.4f}")
    return metrics

---
## 5 – Method A: Variational (TV Regularisation)

Solve the variational problem with Total Variation regularisation using IPPy's solver.

> **TODO:** implement TV reconstruction.  
> Choose the regularisation parameter λ heuristically and document your choice.

In [ ]:
# ── TV parameters ────────────────────────────────────────────────────────────
LAMBDA_TV   = 1e-3     # TODO: tune this parameter
MAX_ITER_TV = 500      # TODO: adjust if needed


def reconstruct_tv(y: np.ndarray, sigma: float) -> np.ndarray:
    """
    Reconstruct x from degraded observation y using TV regularisation.

    Uses IPPy's solver. Adapt the call to the actual IPPy API.

    Example (adapt as needed):
        solver = solvers.TVSolver(K, lam=LAMBDA_TV, max_iter=MAX_ITER_TV)
        x_rec  = solver.solve(y)
    """
    # TODO: implement TV reconstruction with IPPy
    x_rec = None   # placeholder
    return x_rec


# ── Evaluate ─────────────────────────────────────────────────────────────────
# TODO: uncomment once reconstruct_tv and data are ready
# results["TV"] = evaluate_on_test("TV", reconstruct_tv, NOISE_LEVELS, x_test, y_test)

---
## 6 – Method B: End-to-End Neural Network

Select **one** architecture: UNet, ViT, or NAF-Net.

> **TODO:** implement training and inference of the chosen end-to-end network.  
> Justify your architecture choice with respect to the task properties.

In [ ]:
# ── Model definition ─────────────────────────────────────────────────────────
# TODO: define / import your chosen architecture (UNet / ViT / NAF-Net)

# ── Training ─────────────────────────────────────────────────────────────────
# TODO: training loop
#   - use x_train and x_val
#   - apply degrade() on-the-fly or pre-compute y_train
#   - choose and justify: loss, optimiser, lr schedule, batch size, epochs

# ── Inference ─────────────────────────────────────────────────────────────────
def reconstruct_nn(y: np.ndarray, sigma: float) -> np.ndarray:
    """
    Reconstruct x from degraded observation y with the trained neural network.
    """
    # TODO: implement NN inference
    x_rec = None   # placeholder
    return x_rec


# ── Evaluate ──────────────────────────────────────────────────────────────────
# TODO: uncomment once the network is trained
# results["NN"] = evaluate_on_test("NN", reconstruct_nn, NOISE_LEVELS, x_test, y_test)

---
## 7 – Method C: Generative (DiffPIR)

> **TODO:** implement DiffPIR for motion deblurring.  
> Choose and document: number of diffusion steps, noise schedule, and any other hyper-parameters.

In [ ]:
# ── DiffPIR parameters ────────────────────────────────────────────────────────
DIFFPIR_STEPS = 100     # TODO: choose the number of diffusion steps


def reconstruct_diffpir(y: np.ndarray, sigma: float) -> np.ndarray:
    """
    Reconstruct x from y using DiffPIR.
    """
    # TODO: implement DiffPIR
    x_rec = None   # placeholder
    return x_rec


# ── Evaluate ──────────────────────────────────────────────────────────────────
# TODO: uncomment once DiffPIR is implemented
# results["DiffPIR"] = evaluate_on_test("DiffPIR", reconstruct_diffpir, NOISE_LEVELS, x_test, y_test)

---
## 8 – Method D (Optional): Plug-and-Play HQS

Hybrid method: Plug-and-Play with Half Quadratic Splitting, using an IPPy solver.

> **TODO:** implement PnP-HQS.  
> Choose the denoiser (e.g. DnCNN, BM3D) and the HQS parameters (ρ, number of outer iterations).  
> This method can be skipped if the group has only 2 students and has already implemented the other three.

In [ ]:
# ── PnP-HQS parameters ────────────────────────────────────────────────────────
RHO_HQS      = 0.1    # TODO: tune
OUTER_ITER   = 30     # TODO: tune


def reconstruct_pnp(y: np.ndarray, sigma: float) -> np.ndarray:
    """
    Reconstruct x from y with PnP-HQS.

    Example with IPPy (adapt to actual API):
        denoiser = <your_denoiser_function>
        solver   = solvers.HQS(K, denoiser, rho=RHO_HQS, max_iter=OUTER_ITER)
        x_rec    = solver.solve(y)
    """
    # TODO: implement PnP-HQS with IPPy
    x_rec = None   # placeholder
    return x_rec


# ── Evaluate ──────────────────────────────────────────────────────────────────
# TODO: uncomment once implemented
# results["PnP-HQS"] = evaluate_on_test("PnP-HQS", reconstruct_pnp, NOISE_LEVELS, x_test, y_test)

---
## 9 – Comparison & Plots

> **TODO:** generate the comparison plots once all methods are evaluated.

Produce:
1. A **PSNR / SSIM vs. noise level** plot for all methods.
2. A **visual comparison** grid: for each noise level, show clean / degraded / reconstructions side by side.

In [ ]:
# ── Quantitative comparison table ─────────────────────────────────────────────
# TODO: print or display a summary table (methods × noise levels × {PSNR, SSIM})
# Example:
#
# import pandas as pd
# rows = []
# for method, metrics in results.items():
#     for sigma, vals in metrics.items():
#         rows.append({"Method": method, "σ": sigma, **vals})
# df = pd.DataFrame(rows)
# display(df.pivot_table(index="Method", columns="σ", values=["PSNR", "SSIM"]))

In [ ]:
# ── PSNR comparison plot ───────────────────────────────────────────────────────
# TODO: line plot – x = noise level, y = PSNR, one line per method
# Example:
#
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# for method, metrics in results.items():
#     sigmas = sorted(metrics.keys())
#     axes[0].plot(sigmas, [metrics[s]["PSNR"] for s in sigmas], marker="o", label=method)
#     axes[1].plot(sigmas, [metrics[s]["SSIM"] for s in sigmas], marker="o", label=method)
# axes[0].set(xlabel="Noise level σ", ylabel="PSNR (dB)", title="PSNR vs noise level")
# axes[1].set(xlabel="Noise level σ", ylabel="SSIM",     title="SSIM vs noise level")
# for ax in axes: ax.legend(); ax.grid(True)
# plt.tight_layout()
# plt.savefig(RESULTS_DIR / "comparison_metrics.pdf"); plt.show()

In [ ]:
# ── Visual comparison grid ────────────────────────────────────────────────────
# TODO: for each noise level, display a grid:
#   columns = [Clean, Degraded, TV, NN, DiffPIR, PnP-HQS (optional)]
#   rows    = one per selected test image
# Save figures to RESULTS_DIR.

---
## 10 – Discussion

> **TODO (for the oral presentation):** fill in the discussion section.

For each method address:
- **Parameter selection:** how did you choose the key hyper-parameters? (λ for TV, architecture/loss for NN, diffusion steps for DiffPIR, ρ/denoiser for PnP)
- **Strengths:** when does this method perform best?
- **Limitations:** failure modes, computational cost, sensitivity to noise level.
- **Comparison:** which method wins on PSNR/SSIM? Does the ranking change across noise levels? Do PSNR/SSIM align with visual quality?